# PAQFE-v2 vs Standard QML — Preliminary 5-Fold Composition Validation

**Fast version for the presentation.** One shuffled 5-fold GroupKFold repeat only (seed 42), using identical composition-level folds for Standard QML and PAQFE-v2.

PAQFE-v2: grouped features → trainable alpha (lr 0.005) → physics entanglement → 3-layer StronglyEntanglingLayers.
Standard QML: standard feature order → Y AngleEmbedding → 3-layer StronglyEntanglingLayers.

Feature and target scaling are fitted inside each training fold only. Each model/fold is trained once.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pennylane as qml
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
N_GROUP_SPLITS = 5
N_EPOCHS_GROUPED = 150
N_QUBITS = 10
N_LAYERS = 3
LR_VQC = 0.02
LR_ALPHA = 0.005
LAMBDA_PRIOR = 0.01
TARGET = 'UTS_MPa'
STANDARD_FEATURE_ORDER = ['xAl','VEC','dHmix','dSmix','delta','Omega','dchi','Mean_Tm','Temperature_C','log10_Strain_rate']
GROUPED_FEATURE_ORDER = ['xAl','VEC','dchi','dHmix','dSmix','Omega','Mean_Tm','delta','Temperature_C','log10_Strain_rate']
COMPOSITION_COLS = ['Al_norm','Co_norm','Cr_norm','Fe_norm','Ni_norm']
STRONG_RELATIONSHIP_DESCRIPTORS = {'delta','Omega','xAl','VEC'}
UPWEIGHT_FACTOR = 1.2


## 1. Load the same Excel workbook (`B_Model_Ready`)

In [ ]:
from google.colab import files
uploaded = files.upload()
file_path = list(uploaded.keys())[0]
df = pd.read_excel(file_path, sheet_name='B_Model_Ready')
required = STANDARD_FEATURE_ORDER + [TARGET] + COMPOSITION_COLS
missing = [c for c in required if c not in df.columns]
if missing: raise ValueError(f'Missing required columns: {missing}')
df[required] = df[required].apply(pd.to_numeric, errors='coerce')
df = df.dropna(subset=required).copy()
df['Composition_Group'] = df[COMPOSITION_COLS].round(3).astype(str).agg('_'.join, axis=1)
print('Observations:', len(df))
print('Unique compositions:', df['Composition_Group'].nunique())


In [ ]:
physics_prior = torch.tensor([UPWEIGHT_FACTOR if f in STRONG_RELATIONSHIP_DESCRIPTORS else 1.0 for f in GROUPED_FEATURE_ORDER], dtype=torch.float64)
QUBIT_BLOCKS = {'composition':[0,1,2], 'thermodynamic':[3,4,5,6], 'structural':[7], 'context':[8,9]}

def ring_entangle(wires):
    if len(wires) < 2: return
    for i in range(len(wires)):
        qml.CNOT(wires=[wires[i], wires[(i+1)%len(wires)]])

def physics_entangling_layer():
    ring_entangle(QUBIT_BLOCKS['composition'])
    ring_entangle(QUBIT_BLOCKS['thermodynamic'])
    ring_entangle(QUBIT_BLOCKS['context'])
    fmap = {f:i for i,f in enumerate(GROUPED_FEATURE_ORDER)}
    qml.CNOT(wires=[fmap['delta'], fmap['Omega']])
    qml.CNOT(wires=[fmap['xAl'], fmap['VEC']])


## 2. Model builders

In [ ]:
def build_standard_qml():
    dev=qml.device('default.qubit',wires=N_QUBITS)
    @qml.qnode(dev,interface='torch',diff_method='backprop')
    def circuit(inputs,weights):
        qml.templates.AngleEmbedding(inputs,wires=range(N_QUBITS),rotation='Y')
        qml.templates.StronglyEntanglingLayers(weights,wires=range(N_QUBITS))
        return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]
    shape=qml.StronglyEntanglingLayers.shape(n_layers=N_LAYERS,n_wires=N_QUBITS)
    class Model(nn.Module):
        def __init__(self):
            super().__init__()
            self.weights=nn.Parameter(0.01*torch.randn(shape,dtype=torch.float64))
            self.head=nn.Linear(N_QUBITS,1,dtype=torch.float64)
        def forward(self,x):
            q=torch.stack([torch.stack(circuit(s,self.weights)) for s in x],dim=0)
            return self.head(q).squeeze(-1)
    return Model()

def build_paqfe_v2():
    dev=qml.device('default.qubit',wires=N_QUBITS)
    @qml.qnode(dev,interface='torch',diff_method='backprop')
    def circuit(inputs,weights):
        qml.templates.AngleEmbedding(inputs,wires=range(N_QUBITS),rotation='Y')
        physics_entangling_layer()
        qml.templates.StronglyEntanglingLayers(weights,wires=range(N_QUBITS))
        return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]
    shape=qml.StronglyEntanglingLayers.shape(n_layers=N_LAYERS,n_wires=N_QUBITS)
    class Model(nn.Module):
        def __init__(self):
            super().__init__()
            self.alpha=nn.Parameter(physics_prior.clone())
            self.weights=nn.Parameter(0.01*torch.randn(shape,dtype=torch.float64))
            self.head=nn.Linear(N_QUBITS,1,dtype=torch.float64)
        def forward(self,x):
            theta=x*self.alpha
            q=torch.stack(circuit(theta,self.weights),dim=1)
            return self.head(q).squeeze(-1)
    return Model()


## 3. One 5-fold grouped repeat — same folds for both models

In [ ]:
def run_fold(model_name, train_idx, test_idx):
    order = STANDARD_FEATURE_ORDER if model_name == 'Standard QML' else GROUPED_FEATURE_ORDER
    Xtr_raw=df.iloc[train_idx][order].to_numpy(dtype=np.float64)
    Xte_raw=df.iloc[test_idx][order].to_numpy(dtype=np.float64)
    ytr_raw=df.iloc[train_idx][TARGET].to_numpy(dtype=np.float64)
    yte_raw=df.iloc[test_idx][TARGET].to_numpy(dtype=np.float64)
    xs=StandardScaler(); Xtr=xs.fit_transform(Xtr_raw); Xte=xs.transform(Xte_raw)
    ys=StandardScaler(); ytr=ys.fit_transform(ytr_raw.reshape(-1,1)).ravel()
    np.random.seed(SEED); torch.manual_seed(SEED)
    model=build_standard_qml() if model_name=='Standard QML' else build_paqfe_v2()
    if model_name=='Standard QML':
        opt=torch.optim.Adam(model.parameters(),lr=LR_VQC)
    else:
        opt=torch.optim.Adam([
            {'params':model.alpha,'lr':LR_ALPHA},
            {'params':model.weights,'lr':LR_VQC},
            {'params':model.head.parameters(),'lr':LR_VQC}
        ])
    Xtr_t=torch.tensor(Xtr,dtype=torch.float64); ytr_t=torch.tensor(ytr,dtype=torch.float64); Xte_t=torch.tensor(Xte,dtype=torch.float64)
    loss_fn=nn.MSELoss(); model.train()
    for _ in range(N_EPOCHS_GROUPED):
        opt.zero_grad(); pred=model(Xtr_t); loss=loss_fn(pred,ytr_t)
        if model_name=='PAQFE-v2':
            loss=loss + LAMBDA_PRIOR*torch.mean((model.alpha-physics_prior)**2)
        loss.backward(); opt.step()
    model.eval()
    with torch.no_grad(): pred_std=model(Xte_t).cpu().numpy()
    pred=ys.inverse_transform(pred_std.reshape(-1,1)).ravel()
    return yte_raw,pred

splits=list(GroupKFold(n_splits=N_GROUP_SPLITS,shuffle=True,random_state=SEED).split(df[STANDARD_FEATURE_ORDER],df[TARGET],df['Composition_Group']))
records=[]
for fold,(train_idx,test_idx) in enumerate(splits,1):
    for model_name in ['Standard QML','PAQFE-v2']:
        yt,yp=run_fold(model_name,train_idx,test_idx)
        records.append({'Fold':fold,'Model':model_name,'MAE':mean_absolute_error(yt,yp),'RMSE':np.sqrt(mean_squared_error(yt,yp)),'R2':r2_score(yt,yp),'N_Test':len(test_idx)})
        print(f'Fold {fold}/5 | {model_name} complete')

fold_results=pd.DataFrame(records)
display(fold_results)


In [ ]:
summary=fold_results.groupby('Model')[['MAE','RMSE','R2']].agg(['mean','std'])
display(summary)

print('\nPresentation note: this is one 5-fold shuffled composition-level repeat (seed 42), not the final 5-repeat estimate.')
fold_results.to_csv('preliminary_GroupKFold_fold_results.csv',index=False)
summary.to_csv('preliminary_GroupKFold_summary.csv')


## Stop here
Use this as a **preliminary unseen-composition comparison** for tomorrow. The full 5-repeat × 5-fold GroupKFold can be run later for the final paper.